In [0]:
# Import required libraries
from pyspark.sql.functions import col
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Load Data
df = spark.table("data_processed.credit_risk_main.loans_transformed")

Bi-variate Analysis

In [0]:
# ------------------------
# Identify numeric columns
# ------------------------
numeric_cols = ['default_ind', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'emp_length_num', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'annual_inc_joint', 'dti_joint', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_il_6m', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m']

print(f"Numeric columns found: {numeric_cols}")


In [0]:
# ------------------------
# Convert numeric columns to Pandas (sample for performance)
# ------------------------
pdf = df.select(*numeric_cols).sample(fraction=0.1, seed=42).toPandas()

In [0]:
# ------------------------
# 4️⃣ Plot correlation heatmap
# ------------------------
plt.figure(figsize=(24, 20))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

In [0]:
# Round for readability
corr_table = corr_matrix.round(2)

# Display table
display(corr_table)

Scatterplots:

In [0]:
# ------------------------
# Define columns for scatterplots
# ------------------------
scatter_cols = ["annual_inc", "loan_amnt", "revol_bal", "revol_util"]

# Filter only existing columns
existing_cols = [c for c in scatter_cols if c in df.columns]
print(f"Columns found in dataset: {existing_cols}")

In [0]:
# ------------------------
# Convert to Pandas (sample for performance)
# ------------------------
pdf = df.select(*existing_cols).sample(fraction=0.1, seed=42).toPandas()

In [0]:
# ------------------------
# Scatterplot: annual_inc vs loan_amnt
# ------------------------
if "annual_inc" in pdf.columns and "loan_amnt" in pdf.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=pdf, x="annual_inc", y="loan_amnt", alpha=0.5)
    plt.title("Annual Income vs Loan Amount")
    plt.xlabel("Annual Income")
    plt.ylabel("Loan Amount")
    plt.ylim(0, pdf["loan_amnt"].quantile(0.99))  # zoom in to 99th percentile to avoid outlier distortion
    plt.xlim(0, pdf["annual_inc"].quantile(0.99))
    plt.show()

In [0]:
# ------------------------
# Scatterplot: revol_bal vs revol_util
# ------------------------
if "revol_bal" in pdf.columns and "revol_util" in pdf.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=pdf, x="revol_bal", y="revol_util", alpha=0.5)
    plt.title("Revolving Balance vs Revolving Utilization")
    plt.xlabel("Revolving Balance")
    plt.ylabel("Revolving Utilization (%)")
    plt.ylim(0, 100)  # revol_util is typically percentage
    plt.show()

Let’s enhance the scatterplots with regression lines and log scales to better visualize relationships with skewed distributions.

In [0]:
# ------------------------
# Define columns for scatterplots
# ------------------------
scatter_cols = ["annual_inc", "loan_amnt", "revol_bal", "revol_util"]
existing_cols = [c for c in scatter_cols if c in df.columns]

In [0]:
# ------------------------
# Convert to Pandas (sample for performance)
# ------------------------
pdf = df.select(*existing_cols).sample(fraction=0.1, seed=42).toPandas()

In [0]:
# ------------------------
# Scatterplot: annual_inc vs loan_amnt (log scale + regression)
# ------------------------
if "annual_inc" in pdf.columns and "loan_amnt" in pdf.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=pdf, x="annual_inc", y="loan_amnt", alpha=0.5)
    sns.regplot(data=pdf, x="annual_inc", y="loan_amnt", scatter=False, color='red', line_kws={'linewidth':2})
    plt.title("Annual Income vs Loan Amount")
    plt.xlabel("Annual Income")
    plt.ylabel("Loan Amount")
    plt.xscale('log')
    plt.yscale('log')
    plt.show()

In [0]:
# ------------------------
# Scatterplot: revol_bal vs revol_util (regression line)
# ------------------------
if "revol_bal" in pdf.columns and "revol_util" in pdf.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=pdf, x="revol_bal", y="revol_util", alpha=0.5)
    sns.regplot(data=pdf, x="revol_bal", y="revol_util", scatter=False, color='red', line_kws={'linewidth':2})
    plt.title("Revolving Balance vs Revolving Utilization")
    plt.xlabel("Revolving Balance")
    plt.ylabel("Revolving Utilization (%)")
    plt.xscale('log')  # Revolving balance is heavily skewed
    plt.ylim(0, 100)  # revol_util is percentage
    plt.show()

Features:

Regression line (red) overlaid on scatterplots to see the trend.

Log scale for highly skewed features (annual_inc and revol_bal) for better visualization.

Alpha blending (alpha=0.5) to handle overlapping points.

Automatic column checks prevent errors if a column is missing.

Works with a sampled Pandas DataFrame for large datasets.

Default Analysis

In [0]:
# ------------------------
# Step 1: Select relevant columns
# ------------------------
default_cols = ["loan_amnt", "int_rate", "annual_inc", "grade", "term", "home_ownership", "default_ind"]
existing_cols = [c for c in default_cols if c in df.columns]

In [0]:
# Sample for performance
pdf = df.select(*existing_cols).sample(fraction=0.1, seed=42).toPandas()

# Ensure default_ind is integer/binary
pdf["default_ind"] = pdf["default_ind"].astype(int)

In [0]:
# ------------------------
# Step 2: Distribution comparisons
# ------------------------
dist_cols = ["loan_amnt", "int_rate", "annual_inc"]

for col in dist_cols:
    if col in pdf.columns:
        plt.figure(figsize=(8, 6))
        sns.kdeplot(data=pdf, x=col, hue="default_ind", common_norm=False, fill=True, alpha=0.5)
        plt.title(f"Distribution of {col} by Default Indicator")
        plt.xlabel(col)
        plt.ylabel("Density")
        
        # Log scale for skewed features
        if col in ["annual_inc", "loan_amnt"]:
            plt.xscale("log")
        
        plt.show()

In [0]:
# ------------------------
# Step 3: Default rates by grade, term, and home_ownership
# ------------------------
group_features = ["grade", "term", "home_ownership"]

for feature in group_features:
    if feature in pdf.columns:
        rate_df = (
            pdf.groupby(feature)["default_ind"]
            .mean()
            .reset_index()
            .sort_values("default_ind", ascending=False)
        )
        
        plt.figure(figsize=(8, 6))
        sns.barplot(data=rate_df, x=feature, y="default_ind", palette="viridis")
        plt.title(f"Default Rate by {feature}")
        plt.ylabel("Default Rate")
        plt.xlabel(feature)
        plt.show()